In [ ]:
!pip install soundfile

import os
import requests
import warnings
import json
import soundfile as sf
import pandas as pd
from pathlib import Path
from datasets import load_dataset
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from datasets import Dataset as RagasDataset

from ragas import aevaluate
from ragas.metrics import (
    faithfulness,
    answer_relevancy,
    context_precision,
    context_recall,
)

from retrieval.llm import LLMConfig
from ingestion.image_captioner import VLMConfig
from embedding.embedder import EmbedderConfig
from core.rag import RAG

warnings.filterwarnings("ignore", category=DeprecationWarning)
load_dotenv()

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload
Workspace 'evaluation_text_large2' initialized successfully.


In [ ]:
print("Loading TED-LIUM dataset...")
tedlium = load_dataset("LIUM/tedlium", "release1")
samples = [tedlium["train"][i] for i in range(20)]

AUDIO_DIR = Path("./converted_audio")
AUDIO_DIR.mkdir(parents=True, exist_ok=True)

audio_paths = []
transcripts = []

print("Converting .sph files to .wav format...")
for i, sample in enumerate(samples):
    sph_path = sample["file"]
    speaker_id = sample["speaker_id"]
    wav_filename = f"talk_{speaker_id}_{i}.wav"
    wav_path = AUDIO_DIR / wav_filename
    
    data, samplerate = sf.read(sph_path)
    sf.write(wav_path, data, samplerate)
    
    audio_paths.append(str(wav_path))
    transcripts.append(sample["text"])

print(f"Successfully converted and saved {len(audio_paths)} .wav files.")

In [ ]:
qa_llm = ChatOpenAI(
    model="gpt-4o-mini",
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL")
)

generated_queries = []

print("Generating 50 synthetic QA pairs (25 extractive, 25 abstractive)...")

for i in range(25):
    context_text = transcripts[i % len(transcripts)]
    corresponding_audio = audio_paths[i % len(audio_paths)]
    
    prompt = f"Based ONLY on the following text chunk, generate a clear, fact-based question and its explicit answer. Ensure the answer is directly found in the text. Return your response as a JSON object with the keys 'query' and 'answer'.\n\nText: {context_text}"
    
    response = qa_llm.invoke(prompt)
    try:
        clean_content = response.content.strip().strip("```json").strip("```")
        qa_data = json.loads(clean_content)
        generated_queries.append({
            "query": qa_data["query"],
            "answer": qa_data["answer"],
            "source": "text",
            "type": "extractive",
            "pdf_filename": corresponding_audio
        })
    except Exception as e:
        print(f"Failed parsing extractive row {i}: {e}")

for i in range(25):
    context_text = transcripts[i % len(transcripts)]
    corresponding_audio = audio_paths[i % len(audio_paths)]
    
    prompt = f"Based on the following text chunk, generate a high-level conceptual question and its comprehensive answer. The question should require summarizing, synthesizing, or inferring meaning from the context rather than copying explicit strings. Return your response as a JSON object with the keys 'query' and 'answer'.\n\nText: {context_text}"
    
    response = qa_llm.invoke(prompt)
    try:
        clean_content = response.content.strip().strip("```json").strip("```")
        qa_data = json.loads(clean_content)
        generated_queries.append({
            "query": qa_data["query"],
            "answer": qa_data["answer"],
            "source": "text",
            "type": "abstractive",
            "pdf_filename": corresponding_audio
        })
    except Exception as e:
        print(f"Failed parsing abstractive row {i}: {e}")

df_queries = pd.DataFrame(generated_queries)
df_queries.to_csv("queries.csv", index=False)
print(f"Saved {len(df_queries)} queries to queries.csv")

Selected 20 PDFs containing 50 'None' queries.

Adding documents to workspace 'evaluation_text_large2'...
Ingesting: /Users/ivan/Code/retrieva/notebooks/evaluation/storage/evaluation-text-large2/6bd45b2a-23a1-4d92-82a8-374aa0320d1c.pdf
=== Document parser messages ===
Using Tesseract for OCR processing.
OCR on page.number=17/18.
OCR on page.number=27/28.

Processing page 0/29
Processing page 1/29
Processing page 2/29
Processing page 3/29
Processing page 4/29
Processing page 5/29
Processing page 6/29
Processing page 7/29
Processing page 8/29
Processing page 9/29
Processing page 10/29
Processing page 11/29
Processing page 12/29
Processing page 13/29
Processing page 14/29
Processing page 15/29
Processing page 16/29
Processing page 17/29
Processing page 18/29
Processing page 19/29
Processing page 20/29
Processing page 21/29
Processing page 22/29
Processing page 23/29
Processing page 24/29
Processing page 25/29
Processing page 26/29
Processing page 27/29
Processing page 28/29
Ingesting: /Us

In [ ]:
summary_df = pd.concat([means_ext], ignore_index=True)

print("Overall Summary:")
display(summary_df)

Overall Summary:


,run_name,context_precision,context_recall,faithfulness,answer_relevancy
0,reranker,0.8637,0.7083,0.7563,0.7657
